# Sample execution

To keep this run practical, this copy trains for one iteration on the notebook?s first 1,000 training images and evaluates the first 1,000 test images. The full settings remain in the original and commented notebooks. These outputs are a quick demonstration, not the full experiment results.


# Upgrading our MNIST Network

In [1]:
import numpy as np, sys
np.random.seed(1)

from keras.datasets import mnist

(x_train, y_train), (x_test, y_test) = mnist.load_data()

images, labels = (x_train[0:1000].reshape(1000,28*28) / 255,
                  y_train[0:1000])


one_hot_labels = np.zeros((len(labels),10))
for i,l in enumerate(labels):
    one_hot_labels[i][l] = 1
labels = one_hot_labels

test_images = x_test[:1000].reshape(1000,28*28) / 255
test_labels = np.zeros((len(test_images),10))
for i,l in enumerate(y_test[:len(test_images)]):
    test_labels[i][l] = 1

def tanh(x):
    return np.tanh(x)

def tanh2deriv(output):
    return 1 - (output ** 2)

def softmax(x):
    temp = np.exp(x)
    return temp / np.sum(temp, axis=1, keepdims=True)

alpha, iterations = (2, 1)
pixels_per_image, num_labels = (784, 10)
batch_size = 128

input_rows = 28
input_cols = 28

kernel_rows = 3
kernel_cols = 3
num_kernels = 16

hidden_size = ((input_rows - kernel_rows) * 
               (input_cols - kernel_cols)) * num_kernels

# weights_0_1 = 0.02*np.random.random((pixels_per_image,hidden_size))-0.01
kernels = 0.02*np.random.random((kernel_rows*kernel_cols,
                                 num_kernels))-0.01

weights_1_2 = 0.2*np.random.random((hidden_size,
                                    num_labels)) - 0.1



def get_image_section(layer,row_from, row_to, col_from, col_to):
    section = layer[:,row_from:row_to,col_from:col_to]
    return section.reshape(-1,1,row_to-row_from, col_to-col_from)

for j in range(iterations):
    correct_cnt = 0
    for i in range(int(len(images) / batch_size)):
        batch_start, batch_end=((i * batch_size),((i+1)*batch_size))
        layer_0 = images[batch_start:batch_end]
        layer_0 = layer_0.reshape(layer_0.shape[0],28,28)
        layer_0.shape

        sects = list()
        for row_start in range(layer_0.shape[1]-kernel_rows):
            for col_start in range(layer_0.shape[2] - kernel_cols):
                sect = get_image_section(layer_0,
                                         row_start,
                                         row_start+kernel_rows,
                                         col_start,
                                         col_start+kernel_cols)
                sects.append(sect)

        expanded_input = np.concatenate(sects,axis=1)
        es = expanded_input.shape
        flattened_input = expanded_input.reshape(es[0]*es[1],-1)

        kernel_output = flattened_input.dot(kernels)
        layer_1 = tanh(kernel_output.reshape(es[0],-1))
        dropout_mask = np.random.randint(2,size=layer_1.shape)
        layer_1 *= dropout_mask * 2
        layer_2 = softmax(np.dot(layer_1,weights_1_2))

        for k in range(batch_size):
            labelset = labels[batch_start+k:batch_start+k+1]
            _inc = int(np.argmax(layer_2[k:k+1]) == 
                               np.argmax(labelset))
            correct_cnt += _inc

        layer_2_delta = (labels[batch_start:batch_end]-layer_2)\
                        / (batch_size * layer_2.shape[0])
        layer_1_delta = layer_2_delta.dot(weights_1_2.T) * \
                        tanh2deriv(layer_1)
        layer_1_delta *= dropout_mask
        weights_1_2 += alpha * layer_1.T.dot(layer_2_delta)
        l1d_reshape = layer_1_delta.reshape(kernel_output.shape)
        k_update = flattened_input.T.dot(l1d_reshape)
        kernels -= alpha * k_update
    
    test_correct_cnt = 0

    for i in range(len(test_images)):

        layer_0 = test_images[i:i+1]
#         layer_1 = tanh(np.dot(layer_0,weights_0_1))
        layer_0 = layer_0.reshape(layer_0.shape[0],28,28)
        layer_0.shape

        sects = list()
        for row_start in range(layer_0.shape[1]-kernel_rows):
            for col_start in range(layer_0.shape[2] - kernel_cols):
                sect = get_image_section(layer_0,
                                         row_start,
                                         row_start+kernel_rows,
                                         col_start,
                                         col_start+kernel_cols)
                sects.append(sect)

        expanded_input = np.concatenate(sects,axis=1)
        es = expanded_input.shape
        flattened_input = expanded_input.reshape(es[0]*es[1],-1)

        kernel_output = flattened_input.dot(kernels)
        layer_1 = tanh(kernel_output.reshape(es[0],-1))
        layer_2 = np.dot(layer_1,weights_1_2)

        test_correct_cnt += int(np.argmax(layer_2) == 
                                np.argmax(test_labels[i:i+1]))
    if(j % 1 == 0):
        sys.stdout.write("\n"+ \
         "I:" + str(j) + \
         " Test-Acc:"+str(test_correct_cnt/float(len(test_images)))+\
         " Train-Acc:" + str(correct_cnt/float(len(images))))

       0/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0s/step

   16384/11490434 ━━━━━━━━━━━━━━━━━━━━ 2:01 11us/step

   32768/11490434 ━━━━━━━━━━━━━━━━━━━━ 1:28 8us/step 

   49152/11490434 ━━━━━━━━━━━━━━━━━━━━ 5:00 26us/step

  196608/11490434 ━━━━━━━━━━━━━━━━━━━━ 1:17 7us/step 

  286720/11490434 ━━━━━━━━━━━━━━━━━━━━ 54s 5us/step 

... (60 outputs left out)

11223040/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step

11436032/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 6s 1us/step



I:0 Test-Acc:0.03 Train-Acc:0.055

### My understanding (draft): preparing the images

MNIST images start as 28 ? 28 pixel grids. The notebook scales pixel intensities from 0?255 to 0?1 and creates one-hot labels for the ten digits. It temporarily flattens images when loading them, then reshapes each image back to 28 ? 28 so the convolution code can take small image patches.

After running this cell, I should note the training and test accuracy I see: **[add my result]**.

### My understanding (draft): what the convolution adds

The code extracts overlapping 3 ? 3 patches from each image and applies 16 learned filters to those patches. The same filter weights are reused at every location, so a filter can respond to a pattern wherever it appears. The resulting values pass through a `tanh` activation, then a dense output layer predicts one of the ten digits. Dropout is used in the hidden layer during training.

This notebook demonstrates a convolutional step before digit classification. I should describe what the accuracy output shows and point to the code that extracts image patches and reuses the filters: **[add my observation after running]**.

### Reflection to complete after experimenting

- What does one convolution filter detect in the example? **[my observation]**
- What changed when the image or filter values changed? **[my observation]**
- In my own words, why reuse the same filter weights across the image? **[my explanation]**